In [ ]:
# Conditions



In [9]:
# 1. Greedy Basic (With weights)

# Flawed approach, but runs at O(k * len(S) * len(E))

S = { # len(S) number of towers, what location each tower covers
  1: {1, 2, 4},
  2: {3, 4, 5},
  3: {2, 5, 6},
  4: {1, 6, 7},
  5: {9, 10},
  6: {8, 10},
  7: {11},
  8: {8}
}
E = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11} # Locations
k = 3 # Max number of towers (just use all of them)
W = [6, 7, 8, 6, 7, 8, 6, 7, 8, 6, 100] # Weights for each locations

def greedy_basic(S, E, k, W):
  available_towers = {i for i in range(1, len(S)+1)}
  available_locations = E.copy()
  total_score = 0
  for i in range(k):
    # for each time, pick the one that would give the most
    best_tower_increase = 0
    best_tower = 0
    for tower in available_towers:
      tower_increase = 0
      for loc in S[tower]:
        if loc in available_locations:
          tower_increase += W[loc-1]

      if tower_increase > best_tower_increase:
        best_tower_increase = tower_increase
        best_tower = tower

    print("Best tower: ", best_tower, " with an increase of ", best_tower_increase)
    available_towers.discard(best_tower)
    for loc in S[best_tower]:
      available_locations.discard(loc)
    total_score += best_tower_increase
  return total_score

print(greedy_basic(S, E, k, W))

Best tower:  7  with an increase of  100
Best tower:  3  with an increase of  22
Best tower:  2  with an increase of  14
136


In [ ]:
# 2: ILP Basic Without Weights

import pulp as p
import timeit

S = {
  1: {1, 2, 4},
  2: {3, 4, 5},
  3: {2, 5, 6},
  4: {1, 6, 7},
  5: {9, 10},
  6: {8, 10},
  7: {11},
  8: {8}
}
E = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11}
k = 3

def no_weights_ilp_basic(S, E, k):
  Lp_prob = p.LpProblem('MCLP With Weights', p.LpMaximize) # initialize the problem itself, give it a name
  start_time = timeit.default_timer() # start time

  # defining x and y based on S and E, respectively (towers & locations)
  x = p.LpVariable.dicts("x", S, cat = "Binary") # x_i is a (binary) indicator function, 1 if used, 0 if unused

  y = p.LpVariable.dicts("y", E, cat = "Binary") # y_i is a (binary) indicator function, 1 if used, 0 if unused

  # Define maximizing function
  Lp_prob += p.lpSum(y[j] for j in E)

  # Limiting number of towers to k
  Lp_prob += p.lpSum(x[i] for i in S) <= k

  # A location can count only if a selected tower covers it
  for j in E:
    Lp_prob += p.lpSum(x[i] for i in S if j in S[i]) >= y[j]

  print(Lp_prob)

  status = Lp_prob.solve()
  solution_time = timeit.default_timer() - start_time

  print(p.LpStatus[status])

  if p.LpStatus[status] == "Optimal":
    print("Selected towers:", [i for i in S if p.value(x[i]) > 0.5])
    print("Covered locations:", [j for j in E if p.value(y[j]) > 0.5])
    print("Total covered:", p.value(Lp_prob.objective))

  print("Solution Time:", solution_time)



with_weights_ilp_basic(S, E, k)

MCLP_With_Weights:
MAXIMIZE
1*y_1 + 1*y_10 + 1*y_11 + 1*y_2 + 1*y_3 + 1*y_4 + 1*y_5 + 1*y_6 + 1*y_7 + 1*y_8 + 1*y_9 + 0.0
SUBJECT TO
_C1: x_1 + x_2 + x_3 + x_4 + x_5 + x_6 + x_7 + x_8 <= 3

_C2: x_1 + x_4 - y_1 >= 0

_C3: x_1 + x_3 - y_2 >= 0

_C4: x_2 - y_3 >= 0

_C5: x_1 + x_2 - y_4 >= 0

_C6: x_2 + x_3 - y_5 >= 0

_C7: x_3 + x_4 - y_6 >= 0

_C8: x_4 - y_7 >= 0

_C9: x_6 + x_8 - y_8 >= 0

_C10: x_5 - y_9 >= 0

_C11: x_5 + x_6 - y_10 >= 0

_C12: x_7 - y_11 >= 0

VARIABLES
0 <= x_1 <= 1 Integer
0 <= x_2 <= 1 Integer
0 <= x_3 <= 1 Integer
0 <= x_4 <= 1 Integer
0 <= x_5 <= 1 Integer
0 <= x_6 <= 1 Integer
0 <= x_7 <= 1 Integer
0 <= x_8 <= 1 Integer
0 <= y_1 <= 1 Integer
0 <= y_10 <= 1 Integer
0 <= y_11 <= 1 Integer
0 <= y_2 <= 1 Integer
0 <= y_3 <= 1 Integer
0 <= y_4 <= 1 Integer
0 <= y_5 <= 1 Integer
0 <= y_6 <= 1 Integer
0 <= y_7 <= 1 Integer
0 <= y_8 <= 1 Integer
0 <= y_9 <= 1 Integer

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/char

/Users/charlie/Desktop/ml-all-phases/.venv/lib/python3.14/site-packages/pulp/pulp.py:1706: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


In [ ]:
# 3: ILP Basic With Weights

import pulp as p
import timeit

S = {
  1: {1, 2, 4},
  2: {3, 4, 5},
  3: {2, 5, 6},
  4: {1, 6, 7},
  5: {9, 10},
  6: {8, 10},
  7: {11},
  8: {8}
}
E = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11}
k = 3
W = [6, 7, 8, 6, 7, 8, 6, 7, 8, 6, 100]

def with_weights_ilp_basic(S, E, k, W):
  if len(W) != len(E):
    raise ValueError("W needs to be the same size as E")
  Lp_prob = p.LpProblem('MCLP With Weights', p.LpMaximize)
  start_time = timeit.default_timer()

  # defining x and y based on S and E, respectively
  x = p.LpVariable.dicts("x", S, cat = "Binary")
  y = p.LpVariable.dicts("y", E, cat = "Binary")
  w_map = dict(zip(sorted(E), W))

  # Define maximizing function
  Lp_prob += p.lpSum(w_map[j] * y[j] for j in E)

  # Limiting number of towers to k
  Lp_prob += p.lpSum(x[i] for i in S) <= k

  # A location can count only if a selected tower covers it
  for j in E:
    Lp_prob += p.lpSum(x[i] for i in S if j in S[i]) >= y[j]

  print(Lp_prob)

  status = Lp_prob.solve()
  solution_time = timeit.default_timer() - start_time

  print(p.LpStatus[status])

  if p.LpStatus[status] == "Optimal":
    print("Selected towers:", [i for i in S if p.value(x[i]) > 0.5])
    print("Covered locations:", [j for j in E if p.value(y[j]) > 0.5])
    print("Total covered:", p.value(Lp_prob.objective))

  print("Solution Time:", solution_time)



with_weights_ilp_basic(S, E, k, W)

MCLP_With_Weights:
MAXIMIZE
6*y_1 + 6*y_10 + 100*y_11 + 7*y_2 + 8*y_3 + 6*y_4 + 7*y_5 + 8*y_6 + 6*y_7 + 7*y_8 + 8*y_9 + 0.0
SUBJECT TO
_C1: x_1 + x_2 + x_3 + x_4 + x_5 + x_6 + x_7 + x_8 <= 3

_C2: x_1 + x_4 - y_1 >= 0

_C3: x_1 + x_3 - y_2 >= 0

_C4: x_2 - y_3 >= 0

_C5: x_1 + x_2 - y_4 >= 0

_C6: x_2 + x_3 - y_5 >= 0

_C7: x_3 + x_4 - y_6 >= 0

_C8: x_4 - y_7 >= 0

_C9: x_6 + x_8 - y_8 >= 0

_C10: x_5 - y_9 >= 0

_C11: x_5 + x_6 - y_10 >= 0

_C12: x_7 - y_11 >= 0

VARIABLES
0 <= x_1 <= 1 Integer
0 <= x_2 <= 1 Integer
0 <= x_3 <= 1 Integer
0 <= x_4 <= 1 Integer
0 <= x_5 <= 1 Integer
0 <= x_6 <= 1 Integer
0 <= x_7 <= 1 Integer
0 <= x_8 <= 1 Integer
0 <= y_1 <= 1 Integer
0 <= y_10 <= 1 Integer
0 <= y_11 <= 1 Integer
0 <= y_2 <= 1 Integer
0 <= y_3 <= 1 Integer
0 <= y_4 <= 1 Integer
0 <= y_5 <= 1 Integer
0 <= y_6 <= 1 Integer
0 <= y_7 <= 1 Integer
0 <= y_8 <= 1 Integer
0 <= y_9 <= 1 Integer

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/ch

/Users/charlie/Desktop/ml-all-phases/.venv/lib/python3.14/site-packages/pulp/pulp.py:1706: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


In [ ]:
# 4: ILP Basic With Weights and Budgeting

# INSTEAD OF k being the constraint,

import pulp as p
import timeit

S = {
  1: {1, 2, 4},
  2: {3, 4, 5},
  3: {2, 5, 6},
  4: {1, 6, 7},
  5: {9, 10},
  6: {8, 10},
  7: {11},
  8: {8}
}
E = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11}
B = 700
W = [6, 7, 8, 6, 7, 8, 6, 7, 8, 6, 100]
C = [29, 100, 0, 200, 1000, 100, 700, 600]

def with_weights_ilp_basic(S, E, B, W, C):
  if len(W) != len(E) or len(C) != len(S):
    raise ValueError("Bad lengths")
  Lp_prob = p.LpProblem('MCLP With Weights', p.LpMaximize)
  start_time = timeit.default_timer()

  # defining x and y based on S and E, respectively
  x = p.LpVariable.dicts("x", S, cat = "Binary")
  y = p.LpVariable.dicts("y", E, cat = "Binary")
  w_map = dict(zip(sorted(E), W))

  # Define maximizing function
  Lp_prob += p.lpSum(w_map[j] * y[j] for j in E)

  # Limiting number of towers to k
  Lp_prob += p.lpSum(C[i-1] * x[i] for i in S) <= B

  # A location can count only if a selected tower covers it
  for j in E:
    Lp_prob += p.lpSum(x[i] for i in S if j in S[i]) >= y[j]

  print(Lp_prob)

  status = Lp_prob.solve()
  solution_time = timeit.default_timer() - start_time

  print(p.LpStatus[status])

  if p.LpStatus[status] == "Optimal":
    print("Selected towers:", [i for i in S if p.value(x[i]) > 0.5])
    print("Covered locations:", [j for j in E if p.value(y[j]) > 0.5])
    print("Total covered:", p.value(Lp_prob.objective))

  print("Solution Time:", solution_time)



with_weights_ilp_basic(S, E, B, W, C)

MCLP_With_Weights:
MAXIMIZE
6*y_1 + 6*y_10 + 100*y_11 + 7*y_2 + 8*y_3 + 6*y_4 + 7*y_5 + 8*y_6 + 6*y_7 + 7*y_8 + 8*y_9 + 0.0
SUBJECT TO
_C1: 29 x_1 + 100 x_2 + 200 x_4 + 1000 x_5 + 100 x_6 + 700 x_7 + 600 x_8
 <= 700

_C2: x_1 + x_4 - y_1 >= 0

_C3: x_1 + x_3 - y_2 >= 0

_C4: x_2 - y_3 >= 0

_C5: x_1 + x_2 - y_4 >= 0

_C6: x_2 + x_3 - y_5 >= 0

_C7: x_3 + x_4 - y_6 >= 0

_C8: x_4 - y_7 >= 0

_C9: x_6 + x_8 - y_8 >= 0

_C10: x_5 - y_9 >= 0

_C11: x_5 + x_6 - y_10 >= 0

_C12: x_7 - y_11 >= 0

VARIABLES
0 <= x_1 <= 1 Integer
0 <= x_2 <= 1 Integer
0 <= x_3 <= 1 Integer
0 <= x_4 <= 1 Integer
0 <= x_5 <= 1 Integer
0 <= x_6 <= 1 Integer
0 <= x_7 <= 1 Integer
0 <= x_8 <= 1 Integer
0 <= y_1 <= 1 Integer
0 <= y_10 <= 1 Integer
0 <= y_11 <= 1 Integer
0 <= y_2 <= 1 Integer
0 <= y_3 <= 1 Integer
0 <= y_4 <= 1 Integer
0 <= y_5 <= 1 Integer
0 <= y_6 <= 1 Integer
0 <= y_7 <= 1 Integer
0 <= y_8 <= 1 Integer
0 <= y_9 <= 1 Integer

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 


In [ ]:
# 5: ILP Basic Generalized

# Same as budgetting but now the cost for each tower depends not on it being that tower but 

import pulp as p
import timeit

S = {
  1: {1, 2, 4},
  2: {3, 4, 5},
  3: {2, 5, 6},
  4: {1, 6, 7},
  5: {9, 10},
  6: {8, 10},
  7: {11},
  8: {8}
}
E = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11}
B = 700
C = [29, 100, 0, 200, 1000, 100, 700, 600]

# Now cost  is (per tower + per location covered)

# Now depend on which tower used
W = {
  1: {1: 6, 2: 7, 4: 4},
  2: {3: 8, 4: 6, 5: 5},
  3: {2: 5, 5: 7, 6: 8},
  4: {1: 3, 6: 6, 7: 6},
  5: {9: 8, 10: 6},
  6: {8: 7, 10: 4},
  7: {11: 100},
  8: {8: 5}
}

#EC[i][j] = c_i(e_j), represents a new, "cost of production". Same dim as W.
EC = {
  1: {1: 5, 2: 5, 4: 20},
  2: {3: 10, 4: 10, 5: 30},
  3: {2: 25, 5: 10, 6: 10},
  4: {1: 40, 6: 30, 7: 10},
  5: {9: 20, 10: 20},
  6: {8: 15, 10: 40},
  7: {11: 50},
  8: {8: 5}
}

def generalized_ilp_basic(S, E, B, W, C, EC):
  if len(C) != len(S) or any(set(W[i]) != S[i] or set(EC[i]) != S[i] for i in S):
    raise ValueError("Bad lengths")
  Lp_prob = p.LpProblem('Generalized MCLP', p.LpMaximize)
  start_time = timeit.default_timer()

  x = p.LpVariable.dicts("x", S, cat = "Binary")
  pairs = [(i, j) for i in S for j in S[i]] # IMPORTAMT
  y = p.LpVariable.dicts("y", pairs, cat = "Binary")

  # Now uses W[i][j] sum now.
  Lp_prob += p.lpSum(W[i][j] * y[(i, j)] for (i, j) in pairs)

  # Now also takes into account serving cost
  Lp_prob += (p.lpSum(C[i-1] * x[i] for i in S)
              + p.lpSum(EC[i][j] * y[(i, j)] for (i, j) in pairs)) <= B

  # each location is served by at most one tower
  for j in E:
    Lp_prob += p.lpSum(y[(i, j)] for i in S if j in S[i]) <= 1

  # CHANGED: tower i can serve j only if tower i is built
  for (i, j) in pairs:
    Lp_prob += y[(i, j)] <= x[i]

  print(Lp_prob)

  status = Lp_prob.solve()
  solution_time = timeit.default_timer() - start_time

  print(p.LpStatus[status])

  if p.LpStatus[status] == "Optimal":
    print("Selected towers:", [i for i in S if p.value(x[i]) > 0.5])
    print("Assignments (tower, location):", [(i, j) for (i, j) in pairs if p.value(y[(i, j)]) > 0.5])
    print("Total weight:", p.value(Lp_prob.objective))

  print("Solution Time:", solution_time)


generalized_ilp_basic(S, E, B, W, C, EC)

/Users/charlie/Desktop/ml-all-phases/.venv/lib/python3.14/site-packages/pulp/pulp.py:1706: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


Generalized_MCLP:
MAXIMIZE
6*y_(1,_1) + 7*y_(1,_2) + 4*y_(1,_4) + 8*y_(2,_3) + 6*y_(2,_4) + 5*y_(2,_5) + 5*y_(3,_2) + 7*y_(3,_5) + 8*y_(3,_6) + 3*y_(4,_1) + 6*y_(4,_6) + 6*y_(4,_7) + 6*y_(5,_10) + 8*y_(5,_9) + 4*y_(6,_10) + 7*y_(6,_8) + 100*y_(7,_11) + 5*y_(8,_8) + 0.0
SUBJECT TO
_C1: 29 x_1 + 100 x_2 + 200 x_4 + 1000 x_5 + 100 x_6 + 700 x_7 + 600 x_8
 + 5 y_(1,_1) + 5 y_(1,_2) + 20 y_(1,_4) + 10 y_(2,_3) + 10 y_(2,_4)
 + 30 y_(2,_5) + 25 y_(3,_2) + 10 y_(3,_5) + 10 y_(3,_6) + 40 y_(4,_1)
 + 30 y_(4,_6) + 10 y_(4,_7) + 20 y_(5,_10) + 20 y_(5,_9) + 40 y_(6,_10)
 + 15 y_(6,_8) + 50 y_(7,_11) + 5 y_(8,_8) <= 700

_C2: y_(1,_1) + y_(4,_1) <= 1

_C3: y_(1,_2) + y_(3,_2) <= 1

_C4: y_(2,_3) <= 1

_C5: y_(1,_4) + y_(2,_4) <= 1

_C6: y_(2,_5) + y_(3,_5) <= 1

_C7: y_(3,_6) + y_(4,_6) <= 1

_C8: y_(4,_7) <= 1

_C9: y_(6,_8) + y_(8,_8) <= 1

_C10: y_(5,_9) <= 1

_C11: y_(5,_10) + y_(6,_10) <= 1

_C12: y_(7,_11) <= 1

_C13: - x_1 + y_(1,_1) <= 0

_C14: - x_1 + y_(1,_2) <= 0

_C15: - x_1 + y_(1,_4